In [ ]:
!pip install wfdb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.2 MB/s eta 0:00:00


In [ ]:
import wfdb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
all_records = ['100','101','102','103','104','105','106','107','108','109',
               '111','112','113','114','115','116','117','118','119','121',
               '122','123','124','200','201','202','203','205','207','208',
               '209','210','212','213','214','215','217','219','220','221',
               '222','223','228','230','231','232','233','234']

test_records = ['101','106','116','119','200','208','215','221','223','234']
train_records = [r for r in all_records if r not in test_records]

print(f"Training patients ({len(train_records)}):", train_records)
print(f"Testing patients ({len(test_records)}):", test_records)

Training patients (38): ['100', '102', '103', '104', '105', '107', '108', '109', '111', '112', '113', '114', '115', '117', '118', '121', '122', '123', '124', '201', '202', '203', '205', '207', '209', '210', '212', '213', '214', '217', '219', '220', '222', '228', '230', '231', '232', '233']
Testing patients (10): ['101', '106', '116', '119', '200', '208', '215', '221', '223', '234']


In [ ]:
window = 130
valid_symbols = ['N', 'A', 'V', 'L', 'R']

def extract_beats_from_records(record_list):
    beats = []
    labels = []
    for record_name in record_list:
        record = wfdb.rdrecord(record_name, pn_dir='mitdb')
        annotation = wfdb.rdann(record_name, 'atr', pn_dir='mitdb')

        for i in range(len(annotation.sample)):
            center = annotation.sample[i]
            symbol = annotation.symbol[i]

            if symbol not in valid_symbols:
                continue
            if center - window < 0 or center + window > len(record.p_signal):
                continue

            beat = record.p_signal[center-window : center+window, 0]
            beats.append(beat)
            labels.append(symbol)

        print(f"Finished {record_name} — running total: {len(beats)} beats")

    return np.array(beats), np.array(labels)

print("=== Extracting TRAINING set (38 patients) ===")
train_beats, train_labels_raw = extract_beats_from_records(train_records)

print("\n=== Extracting TESTING set (10 patients) ===")
test_beats, test_labels_raw = extract_beats_from_records(test_records)

print("\nFINAL SHAPES")
print("Train beats:", train_beats.shape, "| Train labels:", np.unique(train_labels_raw, return_counts=True))
print("Test beats:", test_beats.shape, "| Test labels:", np.unique(test_labels_raw, return_counts=True))

=== Extracting TRAINING set (38 patients) ===
Finished 100 — running total: 2271 beats
Finished 102 — running total: 2374 beats
Finished 103 — running total: 4457 beats
Finished 104 — running total: 4622 beats
Finished 105 — running total: 7189 beats
Finished 107 — running total: 7248 beats
Finished 108 — running total: 9007 beats
Finished 109 — running total: 11535 beats
Finished 111 — running total: 13659 beats
Finished 112 — running total: 16196 beats
Finished 113 — running total: 17984 beats
Finished 114 — running total: 19857 beats
Finished 115 — running total: 21809 beats
Finished 117 — running total: 23343 beats
Finished 118 — running total: 25620 beats
Finished 121 — running total: 27482 beats
Finished 122 — running total: 29956 beats
Finished 123 — running total: 31473 beats
Finished 124 — running total: 33052 beats
Finished 201 — running total: 34905 beats
Finished 202 — running total: 37020 beats
Finished 203 — running total: 39992 beats
Finished 205 — running total: 42637 b

In [ ]:
print("Train beats — mean:", train_beats.mean(), "| std:", train_beats.std())

print("Train beats — min:", train_beats.min(), "| max:", train_beats.max())

print("Test beats — min:", test_beats.min(), "| max:", test_beats.max())

Train beats — mean: -0.32792737994683485 | std: 0.4569122985290073
Train beats — min: -4.45 | max: 3.64
Test beats — min: -5.12 | max: 5.115


In [ ]:
train_labels_bin = np.array([0 if label == 'N' else 1 for label in train_labels_raw])
test_labels_bin = np.array([0 if label == 'N' else 1 for label in test_labels_raw])

print("Train binary distribution:", np.unique(train_labels_bin, return_counts=True))
print("Test binary distribution:", np.unique(test_labels_bin, return_counts=True))

# Normalize using TRAINING set stats only (same rule as before - no test leakage)
mean = train_beats.mean()
std = train_beats.std()

train_beats_norm = (train_beats - mean) / std
test_beats_norm = (test_beats - mean) / std

X_train_v2 = torch.tensor(train_beats_norm, dtype=torch.float32).unsqueeze(1)
X_test_v2 = torch.tensor(test_beats_norm, dtype=torch.float32).unsqueeze(1)
y_train_v2 = torch.tensor(train_labels_bin, dtype=torch.long)
y_test_v2 = torch.tensor(test_labels_bin, dtype=torch.long)

print("X_train_v2 shape:", X_train_v2.shape)
print("X_test_v2 shape:", X_test_v2.shape)

Train binary distribution: (array([0, 1]), array([54531, 20966]))
Test binary distribution: (array([0, 1]), array([20490,  4036]))
X_train_v2 shape: torch.Size([75497, 1, 260])
X_test_v2 shape: torch.Size([24526, 1, 260])


In [ ]:
class ECG_CNN(nn.Module):
    def __init__(self):
        super(ECG_CNN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=8, kernel_size=5)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(in_channels=8, out_channels=16, kernel_size=5)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(16 * 62, 32)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

print("ECG_CNN class defined successfully")

ECG_CNN class defined successfully


In [ ]:
model_v2 = ECG_CNN()

class_counts_v2 = np.bincount(y_train_v2)
class_weights_v2 = torch.tensor([1.0/class_counts_v2[0], 1.0/class_counts_v2[1]], dtype=torch.float32)
class_weights_v2 = class_weights_v2 / class_weights_v2.sum() * 2

print("Class weights:", class_weights_v2)

criterion_v2 = nn.CrossEntropyLoss(weight=class_weights_v2)
optimizer_v2 = optim.Adam(model_v2.parameters(), lr=0.001)

num_epochs = 80

for epoch in range(num_epochs):
    model_v2.train()
    optimizer_v2.zero_grad()

    outputs = model_v2(X_train_v2)
    loss = criterion_v2(outputs, y_train_v2)

    loss.backward()
    optimizer_v2.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} — Loss: {loss.item():.4f}")

Class weights: tensor([0.5554, 1.4446])
Epoch 5/80 — Loss: 0.6700
Epoch 10/80 — Loss: 0.6381


KeyboardInterrupt: 

In [ ]:
model_v2.eval()

with torch.no_grad():
    test_outputs = model_v2(X_test_v2)
    predicted_v2 = torch.argmax(test_outputs, dim=1)

    correct = (predicted_v2 == y_test_v2).sum().item()
    total = y_test_v2.size(0)
    accuracy = correct / total

print(f"Test Accuracy (patient-independent): {accuracy*100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v2, predicted_v2))

print("\nClassification Report:")
print(classification_report(y_test_v2, predicted_v2, target_names=['Normal', 'Abnormal']))

In [ ]:
weights_dict_v2 = {}

for name, param in model_v2.named_parameters():
    weights_dict_v2[name] = param.detach().numpy()

print("conv1 filter #1 (model_v2):", weights_dict_v2['conv1.weight'][0])

import pickle

with open('ecg_cnn_weights_v2.pkl', 'wb') as f:
    pickle.dump(weights_dict_v2, f)

print("\nSaved! Files in current directory:")
!ls -la *.pkl

In [ ]:
from google.colab import files
files.download('ecg_cnn_weights_v2.pkl')

In [ ]:
## Threshold tuning exploration
import torch
import numpy as np
from sklearn.metrics import classification_report

model_v2.eval()
with torch.no_grad():
    probabilities = torch.softmax(model_v2(X_test_v2), dim=1)
abnormal_scores = probabilities[:, 1].numpy()

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    predicted = (abnormal_scores >= threshold).astype(int)
    print(f"\n=== Threshold = {threshold} ===")
    print(classification_report(
        y_test_v2.numpy(), predicted,
        target_names=['Normal', 'Abnormal'],
        zero_division=0
    ))

In [ ]:
# Check training accuracy too — compare against test accuracy
model_v2.eval()
with torch.no_grad():
    train_outputs = model_v2(X_train_v2)
    predicted_train = torch.argmax(train_outputs, dim=1)
    train_correct = (predicted_train == y_train_v2).sum().item()
    train_accuracy = train_correct / y_train_v2.size(0)

print(f"Train Accuracy: {train_accuracy*100:.2f}%")
print(f"Test Accuracy:  97.92%")  # from your last run

In [ ]:
model_v2 = ECG_CNN()

class_counts_v2 = np.bincount(y_train_v2)
class_weights_v2 = torch.tensor([1.0/class_counts_v2[0], 1.0/class_counts_v2[1]], dtype=torch.float32)
class_weights_v2 = class_weights_v2 / class_weights_v2.sum() * 2

criterion_v2 = nn.CrossEntropyLoss(weight=class_weights_v2)
optimizer_v2 = optim.Adam(model_v2.parameters(), lr=0.001)

num_epochs = 120

for epoch in range(num_epochs):
    model_v2.train()
    optimizer_v2.zero_grad()
    outputs = model_v2(X_train_v2)
    loss = criterion_v2(outputs, y_train_v2)
    loss.backward()
    optimizer_v2.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} — Loss: {loss.item():.4f}")

Epoch 10/120 — Loss: 0.6580
Epoch 20/120 — Loss: 0.5755
Epoch 30/120 — Loss: 0.4715
Epoch 40/120 — Loss: 0.3924
Epoch 50/120 — Loss: 0.3253
Epoch 60/120 — Loss: 0.2751


KeyboardInterrupt: 

In [ ]:
model_v2.eval()
with torch.no_grad():
    train_outputs = model_v2(X_train_v2)
    predicted_train = torch.argmax(train_outputs, dim=1)
    train_accuracy = (predicted_train == y_train_v2).sum().item() / y_train_v2.size(0)

    test_outputs = model_v2(X_test_v2)
    probabilities = torch.softmax(test_outputs, dim=1)
abnormal_scores = probabilities[:, 1].numpy()
predicted = (abnormal_scores >= 0.5).astype(int)

print(f"Train Accuracy: {train_accuracy*100:.2f}%")
from sklearn.metrics import classification_report
print("\n=== 120-epoch model, threshold = 0.5 ===")
print(classification_report(
    y_test_v2.numpy(), predicted,
    target_names=['Normal', 'Abnormal'],
    zero_division=0
))

In [ ]:
model_v2.eval()
with torch.no_grad():
    probabilities = torch.softmax(model_v2(X_test_v2), dim=1)
abnormal_scores = probabilities[:, 1].numpy()

from sklearn.metrics import classification_report
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    predicted = (abnormal_scores >= threshold).astype(int)
    print(f"\n=== Threshold = {threshold} (120-epoch model) ===")
    print(classification_report(
        y_test_v2.numpy(), predicted,
        target_names=['Normal', 'Abnormal'],
        zero_division=0
    ))

In [ ]:
weights_dict_v2 = {}

for name, param in model_v2.named_parameters():
    weights_dict_v2[name] = param.detach().numpy()

print("conv1 filter #1 (120-epoch model):", weights_dict_v2['conv1.weight'][0])

import pickle

with open('ecg_cnn_weights_v2_final.pkl', 'wb') as f:
    pickle.dump(weights_dict_v2, f)

print("\nSaved! Files in current directory:")
!ls -la *.pkl

In [ ]:
from google.colab import files
files.download('ecg_cnn_weights_v2_final.pkl')

In [ ]:
import numpy as np
import pickle

# Upload ecg_cnn_weights_fixedpoint.pkl to this Colab session first (Files panel, or files.upload())
with open('ecg_cnn_weights_fixedpoint.pkl', 'rb') as f:
    qw = pickle.load(f)

def quantize_input(x, frac_bits=12):
    return np.clip(np.round(x * (2**frac_bits)), -32768, 32767).astype(np.int64)

activation_min = {'conv1': 1e18, 'conv2': 1e18, 'fc1': 1e18, 'fc2': 1e18}
activation_max = {'conv1': -1e18, 'conv2': -1e18, 'fc1': -1e18, 'fc2': -1e18}

def fixed_point_forward(raw_beat):
    # raw_beat: shape (260,) - RAW mV values, NOT normalized (folding handles that)
    x = quantize_input(raw_beat, frac_bits=12)  # Q4.12

    # --- conv1: input Q4.12 * weight Q2.14 -> product has 12+14=26 frac bits ---
    w1 = qw['conv1.weight'].reshape(8, 5).astype(np.int64)  # (8 filters, 5 taps)
    b1 = qw['conv1.bias'].astype(np.int64)
    out_len1 = 260 - 5 + 1
    conv1_out = np.zeros((8, out_len1), dtype=np.int64)
    for f in range(8):
        for i in range(out_len1):
            acc = np.sum(x[i:i+5] * w1[f]) # Q26 result
            acc = acc + (b1[f] << 12)      # bias is Q2.14, shift to match Q26 (add 12 more frac bits)
            acc = acc >> 12                # rescale back down to Q14 (keep conv1 output in Q14 for next stage)
            conv1_out[f, i] = acc
    conv1_out = np.maximum(conv1_out, 0)  # ReLU
    activation_min['conv1'] = min(activation_min['conv1'], conv1_out.min())
    activation_max['conv1'] = max(activation_max['conv1'], conv1_out.max())
    # maxpool k=2
    pool1_out = conv1_out[:, 0::2][:, :conv1_out.shape[1]//2]
    pool1_out = np.maximum(pool1_out, conv1_out[:, 1::2][:, :pool1_out.shape[1]])

    # --- conv2: input Q14 * weight Q1.15 -> product has 14+15=29 frac bits ---
    w2 = qw['conv2.weight'].astype(np.int64)  # (16, 8, 5)
    b2 = qw['conv2.bias'].astype(np.int64)
    out_len2 = pool1_out.shape[1] - 5 + 1
    conv2_out = np.zeros((16, out_len2), dtype=np.int64)
    for f in range(16):
        for i in range(out_len2):
            acc = np.sum(pool1_out[:, i:i+5] * w2[f])  # sums over 8 channels x 5 taps, Q29
            acc = acc + (b2[f] << 14)  # bias Q15 -> shift to Q29
            acc = acc >> 15            # rescale to Q14 for next stage
            conv2_out[f, i] = acc
    conv2_out = np.maximum(conv2_out, 0)
    activation_min['conv2'] = min(activation_min['conv2'], conv2_out.min())
    activation_max['conv2'] = max(activation_max['conv2'], conv2_out.max())
    pool2_out = conv2_out[:, 0::2][:, :conv2_out.shape[1]//2]
    pool2_out = np.maximum(pool2_out, conv2_out[:, 1::2][:, :pool2_out.shape[1]])

    flat = pool2_out.flatten()  # Q14, length 992

    # --- fc1: Q14 * Q15 -> Q29 ---
    w_fc1 = qw['fc1.weight'].astype(np.int64)
    b_fc1 = qw['fc1.bias'].astype(np.int64)
    fc1_out = (w_fc1 @ flat)
    fc1_out = fc1_out + (b_fc1 << 14)
    fc1_out = fc1_out >> 15
    fc1_out = np.maximum(fc1_out, 0)
    activation_min['fc1'] = min(activation_min['fc1'], fc1_out.min())
    activation_max['fc1'] = max(activation_max['fc1'], fc1_out.max())

    # --- fc2: Q14 * Q15 -> Q29 ---
    w_fc2 = qw['fc2.weight'].astype(np.int64)
    b_fc2 = qw['fc2.bias'].astype(np.int64)
    fc2_out = (w_fc2 @ fc1_out)
    fc2_out = fc2_out + (b_fc2 << 14)
    fc2_out = fc2_out >> 15
    activation_min['fc2'] = min(activation_min['fc2'], fc2_out.min())
    activation_max['fc2'] = max(activation_max['fc2'], fc2_out.max())

    return fc2_out  # [Normal_score, Abnormal_score], still in Q14 - argmax works fine either way

# Run on the full test set
predictions = []
for beat in test_beats:
    scores = fixed_point_forward(beat)
    predictions.append(np.argmax(scores))
predictions = np.array(predictions)

from sklearn.metrics import classification_report, confusion_matrix
print("Fixed-point simulation accuracy:", (predictions == test_labels_bin).mean() * 100, "%")
print(confusion_matrix(test_labels_bin, predictions))
print(classification_report(test_labels_bin, predictions, target_names=['Normal','Abnormal']))

LIMIT = 32767 / 2**14
print()
for layer in ['conv1', 'conv2', 'fc1', 'fc2']:
    lo = activation_min[layer] / 2**14
    hi = activation_max[layer] / 2**14
    status = "WITHIN RANGE" if max(abs(lo), abs(hi)) <= LIMIT else "WOULD OVERFLOW Q2.14"
    print(f"{layer}: min={lo:.4f}  max={hi:.4f}   [{status}]")

NameError: name 'test_labels_bin' is not defined

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print("Fixed-point simulation accuracy:", (predictions == test_labels_bin).mean() * 100, "%")
print(confusion_matrix(test_labels_bin, predictions))
print(classification_report(test_labels_bin, predictions, target_names=['Normal','Abnormal']))

LIMIT = 32767 / 2**14
print()
for layer in ['conv1', 'conv2', 'fc1', 'fc2']:
    lo = activation_min[layer] / 2**14
    hi = activation_max[layer] / 2**14
    status = "WITHIN RANGE" if max(abs(lo), abs(hi)) <= LIMIT else "WOULD OVERFLOW Q2.14"
    print(f"{layer}: min={lo:.4f}  max={hi:.4f}   [{status}]")

Fixed-point simulation accuracy: 98.50362880208759 %
[[20396    94]
 [  273  3763]]
              precision    recall  f1-score   support

      Normal       0.99      1.00      0.99     20490
    Abnormal       0.98      0.93      0.95      4036

    accuracy                           0.99     24526
   macro avg       0.98      0.96      0.97     24526
weighted avg       0.98      0.99      0.98     24526


conv1: min=0.0000  max=13.2570   [WOULD OVERFLOW Q2.14]
conv2: min=0.0000  max=16.4377   [WOULD OVERFLOW Q2.14]
fc1: min=0.0000  max=27.3165   [WOULD OVERFLOW Q2.14]
fc2: min=-12.0884  max=14.1288   [WOULD OVERFLOW Q2.14]


In [ ]:
import numpy as np
import pickle

# Upload ecg_cnn_weights_fixedpoint.pkl to this Colab session first
with open('ecg_cnn_weights_fixedpoint.pkl', 'rb') as f:
    qw = pickle.load(f)

def quantize_input(x, frac_bits=12):
    return np.clip(np.round(x * (2**frac_bits)), -32768, 32767).astype(np.int64)

def sat16(x):
    return np.clip(x, -32768, 32767)

overflow_count = {'conv1': 0, 'conv2': 0, 'fc1': 0, 'fc2': 0}

def fixed_point_forward_v2(raw_beat):
    x = quantize_input(raw_beat, frac_bits=12)

    w1 = qw['conv1.weight'].reshape(8, 5).astype(np.int64)
    b1 = qw['conv1.bias'].astype(np.int64)
    out_len1 = 260 - 5 + 1
    conv1_out = np.zeros((8, out_len1), dtype=np.int64)
    for f in range(8):
        for i in range(out_len1):
            acc = np.sum(x[i:i+5] * w1[f])
            acc = acc + (b1[f] << 12)
            acc = acc >> 16
            conv1_out[f, i] = acc
    overflow_count['conv1'] += int(((conv1_out > 32767) | (conv1_out < -32768)).sum())
    conv1_out = sat16(conv1_out)
    conv1_out = np.maximum(conv1_out, 0)
    pool1_out = conv1_out[:, 0::2][:, :conv1_out.shape[1]//2]
    pool1_out = np.maximum(pool1_out, conv1_out[:, 1::2][:, :pool1_out.shape[1]])

    w2 = qw['conv2.weight'].astype(np.int64)
    b2 = qw['conv2.bias'].astype(np.int64)
    out_len2 = pool1_out.shape[1] - 5 + 1
    conv2_out = np.zeros((16, out_len2), dtype=np.int64)
    for f in range(16):
        for i in range(out_len2):
            acc = np.sum(pool1_out[:, i:i+5] * w2[f])
            acc = acc + (b2[f] << 10)
            acc = acc >> 15
            conv2_out[f, i] = acc
    overflow_count['conv2'] += int(((conv2_out > 32767) | (conv2_out < -32768)).sum())
    conv2_out = sat16(conv2_out)
    conv2_out = np.maximum(conv2_out, 0)
    pool2_out = conv2_out[:, 0::2][:, :conv2_out.shape[1]//2]
    pool2_out = np.maximum(pool2_out, conv2_out[:, 1::2][:, :pool2_out.shape[1]])

    flat = pool2_out.flatten()

    w_fc1 = qw['fc1.weight'].astype(np.int64)
    b_fc1 = qw['fc1.bias'].astype(np.int64)
    fc1_out = (w_fc1 @ flat)
    fc1_out = fc1_out + (b_fc1 << 10)
    fc1_out = fc1_out >> 16
    overflow_count['fc1'] += int(((fc1_out > 32767) | (fc1_out < -32768)).sum())
    fc1_out = sat16(fc1_out)
    fc1_out = np.maximum(fc1_out, 0)

    w_fc2 = qw['fc2.weight'].astype(np.int64)
    b_fc2 = qw['fc2.bias'].astype(np.int64)
    fc2_out = (w_fc2 @ fc1_out)
    fc2_out = fc2_out + (b_fc2 << 9)
    fc2_out = fc2_out >> 14
    overflow_count['fc2'] += int(((fc2_out > 32767) | (fc2_out < -32768)).sum())
    fc2_out = sat16(fc2_out)

    return fc2_out

predictions_v2 = []
for beat in test_beats:
    scores = fixed_point_forward_v2(beat)
    predictions_v2.append(np.argmax(scores))
predictions_v2 = np.array(predictions_v2)

from sklearn.metrics import classification_report, confusion_matrix
print("Corrected fixed-point simulation accuracy:", (predictions_v2 == test_labels_bin).mean() * 100, "%")
print(confusion_matrix(test_labels_bin, predictions_v2))
print(classification_report(test_labels_bin, predictions_v2, target_names=['Normal','Abnormal']))

print()
print("Times each layer's raw result actually exceeded 16-bit range, out of", len(test_beats), "beats x its output size:")
for layer in ['conv1','conv2','fc1','fc2']:
    print(f"  {layer}: {overflow_count[layer]} saturation events")

KeyboardInterrupt: 

In [ ]:
import os
import numpy as np

# 1) the two variables must exist (they vanish after a Colab runtime reset)
for name in ("test_beats", "test_labels_bin"):
    if name not in globals():
        raise NameError(name + " is missing. Re-run the setup / extraction / labels cells "
                        "first, then run this cell again.")

beats  = np.asarray(test_beats, dtype=np.float64)
labels = np.asarray(test_labels_bin).astype(np.int8)
assert beats.ndim == 2 and beats.shape[1] == 260, "expected beats of 260 samples, got " + str(beats.shape)
assert len(beats) == len(labels), "beats and labels differ in length"
if len(beats) != 24526:
    print("WARNING: expected 24,526 test beats, found", len(beats))

# 2) quantize exactly like fixed_point_forward_v2 does: Q4.12, saturating at 16 bits
q_raw   = np.round(beats * 4096)
clipped = int(((q_raw > 32767) | (q_raw < -32768)).sum())
beats_q = np.clip(q_raw, -32768, 32767).astype(np.int16)
print("beats        :", beats_q.shape)
print("Q4.12 min/max:", int(beats_q.min()), int(beats_q.max()), "(limit -32768 .. 32767)")
print("samples that hit the 16-bit limit:", clipped)
print("Normal:", int((labels == 0).sum()), "  Abnormal:", int((labels == 1).sum()))

# 3) cross-check data: your own validated function's scores on 40 beats
rng = np.random.RandomState(7)
check_idx    = np.sort(rng.choice(len(beats), 40, replace=False))
check_scores = np.zeros((0, 2), dtype=np.int64)
if "fixed_point_forward_v2" in globals():
    saved_counts = dict(overflow_count) if "overflow_count" in globals() else None
    check_scores = np.array([fixed_point_forward_v2(test_beats[i]) for i in check_idx],
                            dtype=np.int64)
    if saved_counts is not None:
        overflow_count.update(saved_counts)      # leave your saturation counters untouched
    print("saved fixed_point_forward_v2 scores for", len(check_idx), "beats")
else:
    print("fixed_point_forward_v2 is not defined in this session: skipping the cross-check scores")

# 4) save and download
np.savez_compressed("test_beats_q412.npz", beats_q412=beats_q, labels=labels,
                    check_idx=check_idx, check_scores=check_scores)
print("file test_beats_q412.npz:", round(os.path.getsize("test_beats_q412.npz") / 1e6, 1), "MB")
from google.colab import files
files.download("test_beats_q412.npz")

beats        : (24526, 260)
Q4.12 min/max: -20972 20951 (limit -32768 .. 32767)
samples that hit the 16-bit limit: 0
Normal: 20490   Abnormal: 4036
saved fixed_point_forward_v2 scores for 40 beats
file test_beats_q412.npz: 5.4 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pickle

# Upload ecg_cnn_weights_fixedpoint.pkl to this Colab session first
with open('ecg_cnn_weights_fixedpoint.pkl', 'rb') as f:
    qw = pickle.load(f)

def quantize_input(x, frac_bits=12):
    return np.clip(np.round(x * (2**frac_bits)), -32768, 32767).astype(np.int64)

def sat16(x):
    # real 16-bit signed saturation, applied at EVERY layer this time, not just the end
    return np.clip(x, -32768, 32767)

overflow_count = {'conv1': 0, 'conv2': 0, 'fc1': 0, 'fc2': 0}

def fixed_point_forward_v2(raw_beat):
    x = quantize_input(raw_beat, frac_bits=12)  # Q4.12, unchanged

    # --- conv1: Q4.12 x Q2.14 -> Q26 acc, bias <<12, rescale >>16 -> Q6.10 ---
    w1 = qw['conv1.weight'].reshape(8, 5).astype(np.int64)
    b1 = qw['conv1.bias'].astype(np.int64)
    out_len1 = 260 - 5 + 1
    conv1_out = np.zeros((8, out_len1), dtype=np.int64)
    for f in range(8):
        for i in range(out_len1):
            acc = np.sum(x[i:i+5] * w1[f])
            acc = acc + (b1[f] << 12)
            acc = acc >> 16                       # CHANGED: was >>12, now targets Q10
            conv1_out[f, i] = acc
    overflow_count['conv1'] += int(((conv1_out > 32767) | (conv1_out < -32768)).sum())
    conv1_out = sat16(conv1_out)                  # NEW: real saturation, not just at the end
    conv1_out = np.maximum(conv1_out, 0)           # ReLU
    pool1_out = conv1_out[:, 0::2][:, :conv1_out.shape[1]//2]
    pool1_out = np.maximum(pool1_out, conv1_out[:, 1::2][:, :pool1_out.shape[1]])

    # --- conv2: Q6.10 x Q1.15 -> Q25 acc, bias <<10, rescale >>15 -> Q6.10 ---
    w2 = qw['conv2.weight'].astype(np.int64)
    b2 = qw['conv2.bias'].astype(np.int64)
    out_len2 = pool1_out.shape[1] - 5 + 1
    conv2_out = np.zeros((16, out_len2), dtype=np.int64)
    for f in range(16):
        for i in range(out_len2):
            acc = np.sum(pool1_out[:, i:i+5] * w2[f])
            acc = acc + (b2[f] << 10)             # CHANGED: was <<14
            acc = acc >> 15                        # unchanged numerically, but now relative to Q25 not Q29
            conv2_out[f, i] = acc
    overflow_count['conv2'] += int(((conv2_out > 32767) | (conv2_out < -32768)).sum())
    conv2_out = sat16(conv2_out)
    conv2_out = np.maximum(conv2_out, 0)
    pool2_out = conv2_out[:, 0::2][:, :conv2_out.shape[1]//2]
    pool2_out = np.maximum(pool2_out, conv2_out[:, 1::2][:, :pool2_out.shape[1]])

    flat = pool2_out.flatten()  # Q6.10, length 992

    # --- fc1: Q6.10 x Q1.15 -> Q25 acc, bias <<10, rescale >>16 -> Q7.9 ---
    w_fc1 = qw['fc1.weight'].astype(np.int64)
    b_fc1 = qw['fc1.bias'].astype(np.int64)
    fc1_out = (w_fc1 @ flat)
    fc1_out = fc1_out + (b_fc1 << 10)             # CHANGED: was <<14
    fc1_out = fc1_out >> 16                        # CHANGED: was >>15
    overflow_count['fc1'] += int(((fc1_out > 32767) | (fc1_out < -32768)).sum())
    fc1_out = sat16(fc1_out)
    fc1_out = np.maximum(fc1_out, 0)

    # --- fc2: Q7.9 x Q1.15 -> Q24 acc, bias <<9, rescale >>14 -> Q6.10 ---
    w_fc2 = qw['fc2.weight'].astype(np.int64)
    b_fc2 = qw['fc2.bias'].astype(np.int64)
    fc2_out = (w_fc2 @ fc1_out)
    fc2_out = fc2_out + (b_fc2 << 9)              # CHANGED: was <<14
    fc2_out = fc2_out >> 14                        # CHANGED: was >>15
    overflow_count['fc2'] += int(((fc2_out > 32767) | (fc2_out < -32768)).sum())
    fc2_out = sat16(fc2_out)

    return fc2_out